# Feature Scaling — When & Which Scaler
**Day 65 · Phase 4 · Machine Learning — Theory + Code**

## Objective
- Understand why feature scaling matters and which models need it
- Learn StandardScaler, MinMaxScaler, and RobustScaler — when to use each
- Master the golden rule: **fit on train only, transform both**
- See scaling's real impact on KNN accuracy

## Imports

In [ ]:
# Standard imports for a scaling practice session
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# Reproducibility
np.random.seed(42)

## Theory
**Why scale?** Models that use *distance* (KNN, K-Means, SVM) or *gradients* (linear/logistic regression, neural networks) are sensitive to feature magnitude. A column ranging 20,000–150,000 (income) will drown out a column ranging 18–80 (age).

**The three scalers:**
| Scaler | What it does | Use when |
|---|---|---|
| StandardScaler | mean=0, std=1 | Default: linear/logistic regression, KNN, SVM, PCA |
| MinMaxScaler | bounds to [0, 1] | Neural networks, need bounded values |
| RobustScaler | median + IQR | Data has many outliers |

**The golden rule:** `fit()` learns from the training set only. `transform()` applies the learned transformation to any set. Never `fit` on test data — that's leakage.

## Example 1 — Build a dataset and see the problem

In [ ]:
# Synthetic customer data: age, income, and a "high spender" label
n = 200
age = np.random.randint(18, 80, n)
income = np.random.normal(60000, 15000, n).round(0)
# inject one extreme outlier
income[0] = 980000

high_spender = (income > 60000).astype(int)   # simple target

df = pd.DataFrame({"age": age, "income": income, "high_spender": high_spender})
print(df.describe().round(1))
# Notice: income std is huge compared to age → income will dominate distances

## Example 2 — StandardScaler (the default)

In [ ]:
# Split FIRST — before any scaling
X = df[["age", "income"]]
y = df["high_spender"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# StandardScaler: mean=0, std=1
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)   # fit + transform on TRAIN
X_test_s  = scaler.transform(X_test)        # transform ONLY on test

# Verify: scaled train columns have mean ~0 and std ~1
print("Scaled train means:", X_train_s.mean(axis=0).round(3))
print("Scaled train stds: ", X_train_s.std(axis=0).round(3))

## Example 3 — MinMaxScaler vs RobustScaler

In [ ]:
# MinMaxScaler: everything squeezed into [0, 1]
mm = MinMaxScaler()
X_mm = mm.fit_transform(X_train)
print("MinMax range:", X_mm.min().round(3), "to", X_mm.max().round(3))

# RobustScaler: uses median + IQR, shrugs off the 980k outlier
rb = RobustScaler()
X_rb = rb.fit_transform(X_train)

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, data, title in zip(
    axes,
    [X_train_s, X_mm, X_rb],
    ["StandardScaler", "MinMaxScaler", "RobustScaler"],
):
    ax.scatter(data[:, 0], data[:, 1], alpha=0.5, s=15)
    ax.set_title(title); ax.set_xlabel("age (scaled)"); ax.grid(alpha=0.3)
axes[0].set_ylabel("income (scaled)")
plt.tight_layout()
plt.show()
# RobustScaler's income axis is far less stretched by the outlier

## Practice Exercise — the golden rule in action

In [ ]:
# WRONG: fitting the scaler on the full dataset (leakage)
bad_scaler = StandardScaler()
X_all = bad_scaler.fit_transform(X)   # test info leaks in via fit

# RIGHT: fit on train, transform both
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# Proof of why: compare the scaler learned from train vs the full dataset
print("mean from train only:", scaler.mean_.round(1))
print("mean from full data: ", bad_scaler.mean_.round(1))
# Different values -> the leaky scaler used info it shouldn't have

## Mini Challenge — does scaling really change KNN?

In [ ]:
# Same model, same data — only difference is scaling
knn_raw = KNeighborsClassifier(n_neighbors=5)
knn_raw.fit(X_train, y_train)
print("KNN unscaled accuracy:", round(knn_raw.score(X_test, y_test), 3))

knn_scaled = KNeighborsClassifier(n_neighbors=5)
knn_scaled.fit(X_train_s, y_train)
print("KNN scaled accuracy:  ", round(knn_scaled.score(X_test_s, y_test), 3))
# Without scaling, income's huge range drowns out age -> worse neighbors

## Bonus — the leak-proof Pipeline way

In [ ]:
# Pipeline enforces the correct order automatically
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model",  LogisticRegression())
])
pipe.fit(X_train, y_train)
print("Pipeline test accuracy:", round(pipe.score(X_test, y_test), 3))

# You can tune hyperparameters across the whole pipeline at once:
# GridSearchCV(pipe, {"model__C": [0.1, 1, 10]}, cv=5)

## Summary
- **StandardScaler** (mean 0, std 1) is the default for KNN, SVM, linear/logistic regression, PCA.
- **MinMaxScaler** ([0, 1]) suits neural networks and bounded-value needs.
- **RobustScaler** (median + IQR) is the outlier-resistant choice.
- **Fit on train only, transform both sets** — leakage silently inflates scores.
- Tree-based models (Random Forest, XGBoost) don't need scaling at all.
- Wrap scaler + model in a **Pipeline** to make the correct workflow automatic.